# 기업개요 CSV 4개 통합

1~4페이지를 순서대로 합쳐 `data/기업개요_통합.csv`에 저장합니다. 법인등록번호 등의 앞자리 0을 보존하도록 모든 값을 문자열로 처리하고, 중복 제거 없이 모든 행을 유지합니다. 원본 CSV는 변경하지 않습니다.

In [1]:
import csv
from pathlib import Path

# 프로젝트 루트 또는 하위 폴더에서 실행할 수 있습니다.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / '금융위원회_기업기본정보_수집').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('프로젝트의 금융위원회_기업기본정보_수집 폴더를 찾을 수 없습니다.')

source_dir = project_root / '금융위원회_기업기본정보_수집'
source_files = [
    source_dir / '기업개요_1페이지_10000건.csv',
    source_dir / '기업개요_페이지_2.csv',
    source_dir / '기업개요_페이지_3.csv',
    source_dir / '기업개요_페이지_4.csv',
]
output_file = project_root / 'data' / '기업개요_통합.csv'

# 저장 전에 파일 4개의 컬럼 구성과 행 구조를 확인합니다.
columns = None
merged_rows = []
for source_file in source_files:
    with source_file.open('r', encoding='utf-8-sig', newline='') as file:
        reader = csv.reader(file)
        header = next(reader, None)
        if not header:
            raise ValueError(f'헤더가 없습니다: {source_file.name}')
        if columns is None:
            columns = header
        elif header != columns:
            raise ValueError(f'컬럼 구성이 다릅니다: {source_file.name}')

        row_count = 0
        for row in reader:
            if not row:
                continue
            if len(row) != len(columns):
                raise ValueError(f'행의 컬럼 수가 다릅니다: {source_file.name}, {reader.line_num}행')
            merged_rows.append(row)
            row_count += 1
        print(f'{source_file.name}: {row_count:,}행')

output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(columns)
    writer.writerows(merged_rows)

# 저장된 데이터가 합친 원본 데이터와 같은지 확인합니다.
with output_file.open('r', encoding='utf-8-sig', newline='') as file:
    reader = csv.reader(file)
    assert next(reader) == columns
    assert list(reader) == merged_rows, '저장된 데이터가 원본과 다릅니다.'

print(f'통합 완료: {len(merged_rows):,}행 × {len(columns)}개 컬럼')
print(f'저장 위치: {output_file}')


기업개요_1페이지_10000건.csv: 10,000행
기업개요_페이지_2.csv: 10,000행
기업개요_페이지_3.csv: 10,000행
기업개요_페이지_4.csv: 10,000행
통합 완료: 40,000행 × 37개 컬럼
저장 위치: c:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\data\기업개요_통합.csv


In [5]:
import pandas as pd

df=pd.read_csv('기업개요_통합.csv')
df.columns

C:\Users\Playdata\AppData\Local\Temp\ipykernel_7532\3553099711.py:3: DtypeWarning: Columns (0: enpKrxLstgDt, 1: enpKrxLstgAbolDt) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('기업개요_통합.csv')


Index(['crno', 'corpNm', 'corpEnsnNm', 'enpPbanCmpyNm', 'enpRprFnm',
       'corpRegMrktDcd', 'corpRegMrktDcdNm', 'corpDcd', 'corpDcdNm', 'bzno',
       'enpOzpno', 'enpBsadr', 'enpDtadr', 'enpHmpgUrl', 'enpTlno', 'enpFxno',
       'sicNm', 'enpEstbDt', 'enpStacMm', 'enpXchgLstgDt', 'enpXchgLstgAbolDt',
       'enpKosdaqLstgDt', 'enpKosdaqLstgAbolDt', 'enpKrxLstgDt',
       'enpKrxLstgAbolDt', 'smenpYn', 'enpMntrBnkNm', 'enpEmpeCnt',
       'empeAvgCnwkTermCtt', 'enpPn1AvgSlryAmt', 'actnAudpnNm',
       'audtRptOpnnCtt', 'enpMainBizNm', 'fssCorpUnqNo', 'fssCorpChgDtm',
       'fstOpegDt', 'lastOpegDt'],
      dtype='str')

In [6]:
df1=df.drop(columns=['corpRegMrktDcd','corpRegMrktDcdNm','corpDcd','corpDcdNm','enpOzpno','enpFxno','enpStacMm','enpKosdaqLstgDt','enpKosdaqLstgAbolDt','enpKrxLstgDt','enpKrxLstgAbolDt','smenpYn','enpMntrBnkNm','enpPn1AvgSlryAmt','actnAudpnNm','fssCorpUnqNo','fssCorpChgDtm','fstOpegDt'])
df1.columns

Index(['crno', 'corpNm', 'corpEnsnNm', 'enpPbanCmpyNm', 'enpRprFnm', 'bzno',
       'enpBsadr', 'enpDtadr', 'enpHmpgUrl', 'enpTlno', 'sicNm', 'enpEstbDt',
       'enpXchgLstgDt', 'enpXchgLstgAbolDt', 'enpEmpeCnt',
       'empeAvgCnwkTermCtt', 'audtRptOpnnCtt', 'enpMainBizNm', 'lastOpegDt'],
      dtype='str')

In [8]:
df1.isna().sum()

crno                      0
corpNm                    0
corpEnsnNm            13060
enpPbanCmpyNm         19116
enpRprFnm              9634
bzno                   9937
enpBsadr               7458
enpDtadr              22743
enpHmpgUrl            24018
enpTlno                7307
sicNm                 38295
enpEstbDt             10819
enpXchgLstgDt         35356
enpXchgLstgAbolDt     39521
enpEmpeCnt                0
empeAvgCnwkTermCtt    32561
audtRptOpnnCtt        34285
enpMainBizNm          39738
lastOpegDt                0
dtype: int64

In [10]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   crno                40000 non-null  int64  
 1   corpNm              40000 non-null  str    
 2   corpEnsnNm          26940 non-null  str    
 3   enpPbanCmpyNm       20884 non-null  str    
 4   enpRprFnm           30366 non-null  str    
 5   bzno                30063 non-null  float64
 6   enpBsadr            32542 non-null  str    
 7   enpDtadr            17257 non-null  str    
 8   enpHmpgUrl          15982 non-null  str    
 9   enpTlno             32693 non-null  str    
 10  sicNm               1705 non-null   str    
 11  enpEstbDt           29181 non-null  float64
 12  enpXchgLstgDt       4644 non-null   str    
 13  enpXchgLstgAbolDt   479 non-null    str    
 14  enpEmpeCnt          40000 non-null  int64  
 15  empeAvgCnwkTermCtt  7439 non-null   str    
 16  audtRptOpnnCtt 

In [18]:
df1[['enpXchgLstgDt','enpXchgLstgAbolDt']] = df1[['enpXchgLstgDt','enpXchgLstgAbolDt']].apply(pd.to_datetime, errors='coerce')

C:\Users\Playdata\AppData\Local\Temp\ipykernel_7532\1953989232.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1[['enpXchgLstgDt','enpXchgLstgAbolDt']] = df1[['enpXchgLstgDt','enpXchgLstgAbolDt']].apply(pd.to_datetime, errors='coerce')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_7532\1953989232.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1[['enpXchgLstgDt','enpXchgLstgAbolDt']] = df1[['enpXchgLstgDt','enpXchgLstgAbolDt']].apply(pd.to_datetime, errors='coerce')


In [19]:
df1[['enpXchgLstgDt','enpXchgLstgAbolDt']]

,enpXchgLstgDt,enpXchgLstgAbolDt
0,NaT,NaT
1,NaT,NaT
2,NaT,NaT
3,NaT,NaT
4,NaT,NaT
...,...,...
39995,NaT,NaT
39996,NaT,NaT
39997,NaT,NaT
39998,NaT,NaT


In [ ]:
df1[df1['enpXchgLstgDt']<df1['enpXchgLstgAbolDt']]

df1[(df1['enpXchgLstgDt'].notna()) & (df1['crno']==1101110002818)]
df1[df1['enpXchgLstgDt'].dt.year > 2026]

# df1[df1['crno']==1101110002818]

df[df['crno']==1101110003262]

282    56/03/03
283    56/03/03
284    56/03/03
285    56/03/03
286    56/03/03
287         NaN
288    56/03/03
289    56/03/03
290    56/03/03
291    56/03/03
292    56/03/03
293    56/03/03
294    56/03/03
295    56/03/03
296    56/03/03
Name: enpXchgLstgDt, dtype: str

In [49]:
mask = df1['enpXchgLstgDt'].dt.year > 2026
df1.loc[mask, 'enpXchgLstgDt'] = df1.loc[mask, 'enpXchgLstgDt'] - pd.DateOffset(years=100)
mask2 = df1['enpXchgLstgAbolDt'].dt.year > 2026
df1.loc[mask2, 'enpXchgLstgAbolDt'] = df1.loc[mask2, 'enpXchgLstgAbolDt'] - pd.DateOffset(years=100)

In [54]:

df1=df1.drop(columns=['enpXchgLstgDt','enpXchgLstgAbolDt'])
df1



,crno,corpNm,corpEnsnNm,enpPbanCmpyNm,enpRprFnm,bzno,enpBsadr,enpDtadr,enpHmpgUrl,enpTlno,sicNm,enpEstbDt,enpEmpeCnt,empeAvgCnwkTermCtt,audtRptOpnnCtt,enpMainBizNm,lastOpegDt,상장여부
0,0,SAMPO FUND MANAGEMENT LTD/MANDATUM EMERGING,SAMPO FUND MANAGEMENT LTD/MANDATUM EMERGING,SAMPOFUNDMANAGEMENTLTD/MANDATUMEMERGING,KIMMO LAAKSONEN,NaN,서울특별시 종로구 신문로2가 시티빌딩,NaN,NaN,0220041331,NaN,19990901.0,0,NaN,NaN,NaN,20260914,0
1,0,"ARMOR QUALIFIED, LP","ARMOR QUALIFIED, LP","ARMORQUALIFIED,LP",KRISTINE GUARNERI,NaN,서울특별시 중구 다동39번지,NaN,NaN,212-808-3721,NaN,20060120.0,0,NaN,NaN,NaN,20260914,0
2,0,리만 브라더스,LEHMAN BROTHERS INC,리만브라더스,"RICHARD S.FUND,JR.",NaN,"745 Seventh Avenue New York, NY 10019",NaN,www.lehman.com,1-212-526-7000,NaN,19650121.0,0,NaN,NaN,NaN,20260914,0
3,0,Citigroup Financial Products Inc.,Citigroup Financial Products Inc.,CitigroupFinancialProductsInc.,Robert Druskin,NaN,서울특별시 중구 다동,NaN,NaN,212-816-5605,NaN,19840706.0,0,NaN,NaN,NaN,20260914,0
4,0,"FIREBIRD GLOBAL MASTER FUND, LTD","FIREBIRD GLOBAL MASTER FUND, LTD","FIREBIRDGLOBALMASTERFUND,LTD",JAMES PASSIN,NaN,서울특별시 종로구 신문로2가 시티빌딩,NaN,NaN,0220041331,NaN,20030508.0,0,NaN,NaN,NaN,20260914,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,1101110897269,(주)홍익애이디넷,NaN,NaN,NaN,NaN,전라남도 나주시,동수농공단지길 62-40 ((운곡동) 운곡동),NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,20250318,0
39996,1101110897269,(주)홍익애이디넷,HONG IK,NaN,정원찰,1.088144e+09,서울 강남구 역삼동,603-3 타비쉬빌딩 6층,NaN,82-02-554-0951,NaN,19921022.0,10,NaN,NaN,NaN,20200517,0
39997,1101110897300,(주)명성건축ENG.,MYOUNG SUNG ARCHITECT,NaN,이강억,1.138126e+09,서울 구로구 구로6동,95-6,NaN,82-02-852-8521,NaN,19921022.0,0,NaN,NaN,NaN,20200517,0
39998,1101110897409,(주)연세스포츠센터리즈마트,NaN,NaN,서태순,1.108143e+09,서울 서대문구 홍제동,158-33,NaN,82-02-391-3500,NaN,19991015.0,0,NaN,NaN,NaN,20200517,0


Index(['crno', 'corpNm', 'corpEnsnNm', 'enpPbanCmpyNm', 'enpRprFnm', 'bzno',
       'enpBsadr', 'enpDtadr', 'enpHmpgUrl', 'enpTlno', 'sicNm', 'enpEstbDt',
       'enpEmpeCnt', 'empeAvgCnwkTermCtt', 'audtRptOpnnCtt', 'enpMainBizNm',
       'lastOpegDt', '상장여부'],
      dtype='str')

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path('기업개요전처리.csv')

# crno는 계산용 숫자가 아니라 식별자이므로 문자열로 읽습니다.
df = pd.read_csv(csv_path, dtype={'crno': 'string'})

# CSV 저장 과정에서 생긴 불필요한 인덱스 열이 있으면 제거합니다.
df = df.loc[:, ~df.columns.str.startswith('Unnamed:')]

before_count = len(df)
crno_is_zero = df['crno'].fillna('').str.strip().eq('0')
df = df.loc[~crno_is_zero].reset_index(drop=True)

print(f'삭제한 행 수: {before_count - len(df):,}')
print(f'남은 행 수: {len(df):,}')
assert not df['crno'].fillna('').str.strip().eq('0').any()


삭제한 행 수: 24
남은 행 수: 39,976


,crno,corpNm,corpEnsnNm,enpPbanCmpyNm,enpRprFnm,bzno,enpBsadr,enpDtadr,enpHmpgUrl,enpTlno,sicNm,enpEstbDt,enpEmpeCnt,empeAvgCnwkTermCtt,audtRptOpnnCtt,enpMainBizNm,lastOpegDt,상장여부
0,4108280449,두류야외음악당지역주택조합,NaN,NaN,석자은,4.108280e+09,대구광역시 달서구 달구벌대로 1666 (감삼스퀘어),", 502호,503호",NaN,0535577600,건설업(41-42),NaN,4,NaN,NaN,NaN,20260914,0
1,4108280449,두류야외음악당지역주택조합,NaN,NaN,석자은,4.108280e+09,대구광역시 달서구 달구벌대로 1666 (감삼스퀘어),", 502호,503호",NaN,0535577600,NaN,NaN,4,NaN,NaN,NaN,20250331,0
2,4178401361,(주)라이프엣그,life,NaN,마사추쿠히로시,4.178401e+09,전남 여수시 덕충동,100 (덕충안길),NaN,82-061-6282-3940,NaN,20120512.0,0,NaN,NaN,NaN,20200517,0
3,13000001504,솔라원오호 주식회사,solarone5,솔라원오호,정재훈,6.838804e+09,"전북특별자치도 고창군 흥덕면 부안로 148 (치룡리, 쏠라파크태양광발전소)",NaN,NaN,01*********,NaN,20251022.0,0,NaN,NaN,NaN,20260422,0
4,105020209750,(유)경응,KYUNG EUNG,NaN,강용구,1.050202e+08,대구 달서구 송현동,1990-11,NaN,82-053-656-1341,NaN,NaN,0,NaN,NaN,NaN,20200517,0


In [64]:
df[df['corpNm']=='(주)유수홀딩스']

,crno,corpNm,corpEnsnNm,enpPbanCmpyNm,enpRprFnm,bzno,enpBsadr,enpDtadr,enpHmpgUrl,enpTlno,sicNm,enpEstbDt,enpEmpeCnt,empeAvgCnwkTermCtt,audtRptOpnnCtt,enpMainBizNm,lastOpegDt,상장여부
258,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,23,16년 2개월,적정의견,NaN,20260914,1
259,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,23,16년 2개월,NaN,NaN,20260323,1
260,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,24,14년 9개월,NaN,NaN,20260322,1
261,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,24,14년 9개월,NaN,NaN,20260310,1
262,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)","서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,24,14년 9개월,NaN,NaN,20260101,1
264,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,24,14년 9개월,적정의견,NaN,20251007,1
265,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,23,14년5개월,적정의견,NaN,20250320,1
266,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,23,14년5개월,NaN,NaN,20240320,1
267,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,22,14년 6개월,적정의견,NaN,20240319,1
268,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,25,12년 1개월,적정의견,NaN,20230320,1


In [65]:
# crno가 0인 행을 제외하고, crno별 lastOpegDt가 가장 최신인 행만 남깁니다.
df = df[df['crno'].fillna('').astype('string').str.strip().ne('0')].copy()

df['_lastOpegDt'] = pd.to_datetime(
    df['lastOpegDt'].astype('string').str.strip(),
    format='%Y%m%d',
    errors='coerce'
)

# 날짜가 같은 중복 행은 현재 순서상 첫 번째 행을 남깁니다.
before_count = len(df)
df = (
    df.sort_values(
        ['crno', '_lastOpegDt'],
        ascending=[True, False],
        na_position='last',
        kind='stable'
    )
    .drop_duplicates(subset='crno', keep='first')
    .drop(columns='_lastOpegDt')
    .reset_index(drop=True)
)

print(f'삭제한 중복 행 수: {before_count - len(df):,}')
print(f'최신 행만 남긴 데이터 크기: {df.shape}')
assert not df['crno'].duplicated().any()


삭제한 중복 행 수: 22,587
최신 행만 남긴 데이터 크기: (17389, 18)


In [68]:
df.head()

,crno,corpNm,corpEnsnNm,enpPbanCmpyNm,enpRprFnm,bzno,enpBsadr,enpDtadr,enpHmpgUrl,enpTlno,sicNm,enpEstbDt,enpEmpeCnt,empeAvgCnwkTermCtt,audtRptOpnnCtt,enpMainBizNm,lastOpegDt,상장여부
0,1001110851984,삼성금은 주식회사,"samsunggold&silver co., ltd",삼성금은,이 규석,2.088117e+09,서울특별시 종로구 봉익동 141-1 세화빌딩 211호,NaN,www.samsunggold.com,02-764-2869,NaN,19920523.0,0,NaN,NaN,NaN,20260914,0
1,1001116020799,이지앤스토리 주식회사,"EZNstory Co.,Ltd.",이지앤스토리,"유승민,조현철",1.608700e+09,"서울특별시 구로구 디지털로34길 43 1107호 (구로동, 코오롱싸이언스밸리1차)",NaN,www.eznstory.com,02-6952-3711,NaN,20160406.0,0,NaN,NaN,NaN,20240403,0
2,1001116253720,주식회사 파낙토스,Panaxtos Corp.,파낙토스,박병운,3.628601e+09,"서울특별시 송파구 올림픽로 342 (방이동, 아울타워) 15층",NaN,www.panaxtos.com,02-2051-1380,NaN,20161208.0,0,NaN,NaN,NaN,20231212,0
3,1001140001882,인듐코퍼레이션 코리아(유),"Indium Corporation (Korea)Co.,Ltd",인듐코퍼레이션코리아,그레고리 피에번스,3.018196e+09,"충청북도 청주시 흥덕구 직지대로436번길 24 (송정동, 인듐코퍼레이션)",NaN,www.indiumcom,07077805200,NaN,20080114.0,0,NaN,NaN,NaN,20250920,0
4,1008100969000,동해양조공업(주),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,20260914,0


In [70]:
df.to_csv('기업개요.csv')

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path

input_path = Path(r'C:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\기업개요.csv')
output_path = input_path.with_name('기업개요_대표자정리.csv')

df_rep = pd.read_csv(input_path, dtype={'enpRprFnm': 'string'})
df_rep = df_rep.loc[:, ~df_rep.columns.str.startswith('Unnamed:')]

# 직함은 이름과 붙어 있거나 괄호 안에 있을 수 있으므로 직함만 제거합니다.
role_re = re.compile(
    r'(?<![가-힣])(?:각자\s*대표\s*이사|공동\s*대표\s*이사|단독\s*대표\s*이사|대표\s*집행\s*임원|대표\s*이사|대표이사|대표자|대표|회장|부회장|사장|집행\s*임원|이사)(?![가-힣])',
    flags=re.IGNORECASE,
)
role_word_re = re.compile(r'대표|이사|회장|사장|공동|각자|단독|집행임원')
count_re = re.compile(r'\s*외\s*\d+\s*(?:명|인)?|\s*외\s*$')

def keep_parenthetical(match):
    content = match.group(1).strip()
    # 직함·인원수 설명은 제거하고, 괄호 안의 실제 이름은 보존합니다.
    if role_word_re.search(content) or re.search(r'\d+\s*(?:명|인)', content):
        return ' '
    return f', {content}, '

def clean_representatives(value):
    if pd.isna(value):
        return []
    text = str(value).replace('\n', ' ').strip()
    text = re.sub(r'\(([^()]*)\)', keep_parenthetical, text)
    text = count_re.sub(' ', text)
    text = role_re.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip(' ,')
    if not text:
        return []

    # 쉼표·슬래시·세미콜론·및을 대표자 구분자로 처리합니다.
    parts = re.split(r'\s*(?:,|/|;|·|및)\s*', text)
    names = []
    for part in parts:
        part = re.sub(r'\s+', ' ', part).strip(' ,')
        if not part:
            continue
        # 한국 이름은 성과 이름 사이 공백을 제거합니다.
        if re.fullmatch(r'[가-힣 ]+', part):
            tokens = part.split()
            if len(tokens) > 1 and all(re.fullmatch(r'[가-힣]{2,3}', token) for token in tokens):
                names.extend(tokens)
            else:
                names.append(''.join(tokens))
        else:
            names.append(part)

    return list(dict.fromkeys(names))

# df_rep에서는 실제 파이썬 리스트로 보관합니다.
df_rep['enpRprFnm'] = df_rep['enpRprFnm'].map(clean_representatives)

# CSV에는 리스트를 JSON 문자열로 저장해 재사용할 수 있게 합니다.
csv_df = df_rep.copy()
csv_df['enpRprFnm'] = csv_df['enpRprFnm'].map(lambda names: json.dumps(names, ensure_ascii=False))
csv_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'저장 위치: {output_path}')
print(f'데이터 크기: {df_rep.shape}')
print(df_rep[['enpRprFnm']].head())


In [72]:
import csv
import json
from pathlib import Path

input_path = Path('기업개요_대표자정리.csv')
output_path = input_path.with_suffix('.jsonl')

with input_path.open('r', encoding='utf-8-sig', newline='') as input_file, output_path.open('w', encoding='utf-8', newline='') as output_file:
    reader = csv.DictReader(input_file)
    for row in reader:
        # CSV에 JSON 문자열로 저장된 대표자 리스트를 실제 리스트로 복원합니다.
        row['enpRprFnm'] = json.loads(row['enpRprFnm'])
        # 빈 문자열은 JSON null로 저장합니다.
        row = {key: (None if value == '' else value) for key, value in row.items()}
        output_file.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'저장 위치: {output_path}')


저장 위치: 기업개요_대표자정리.jsonl


In [73]:
df=pd.read_csv("기업개요_대표자정리.csv")


In [75]:
len(df)

17389

In [74]:
df.isna().sum()

crno                      0
corpNm                    0
corpEnsnNm             7964
enpPbanCmpyNm         11281
enpRprFnm                 0
bzno                   6134
enpBsadr               5111
enpDtadr              11471
enpHmpgUrl            13551
enpTlno                6178
sicNm                 17360
enpEstbDt              7182
enpEmpeCnt                0
empeAvgCnwkTermCtt    16763
audtRptOpnnCtt        16811
enpMainBizNm          17382
lastOpegDt                0
상장여부                      0
dtype: int64

In [ ]:


('모기업','계열관계다','모기업')
('모기업','종속관계다','종속기업')
('모기업','위치해있다','지역')
('종속기업','위치해있다','지역')
('종속기업','관련있다','업종')






# 기업 관계 지식그래프 온톨로지

교안의 관계 시그니처, 트리플 추출, 시그니처 검증 흐름을 기업 데이터에 적용한다.

- ParentCompany: 기업개요 CSV의 crno를 식별자로 사용하는 모기업 노드
- SubsidiaryCompany: crno가 없으므로 정규화된 기업명과 주소를 전역 합성 키로 사용하는 노드
- Region: 기업 또는 종속기업의 지역 노드
- Industry: 기업의 업종 또는 종속기업 주요사업내용 노드

입력 파일은 data/clean/기업개요_최종.csv와 data/clean/모기업_계열사_종속기업_통합.csv이다. 기업개요에 존재하는 crno만 ParentCompany로 만들며, 같은 종속기업이 여러 모기업에 연결되면 하나의 SubsidiaryCompany 노드에 여러 HAS_SUBSIDIARY 관계를 연결한다.


In [1]:
from pathlib import Path
import html
import json
import re
import pandas as pd

CORP_FILENAME = "기업개요_최종.csv"
INTEGRATED_FILENAME = "모기업_계열사_종속기업_통합.csv"
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "clean" / CORP_FILENAME).exists()),
    Path.cwd(),
)
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CORP_PATH = CLEAN_DIR / CORP_FILENAME
INTEGRATED_PATH = CLEAN_DIR / INTEGRATED_FILENAME

ONTOLOGY_SCHEMA = {
    "nodes": {
        "ParentCompany": {
            "description": "계열관계 또는 종속관계의 기준이 되는 법인 기업",
            "properties": {
                "crno": {"type":"string", "description":"법인등록번호"},
                "name": {"type":"string", "description":"기업명"},
                "address": {"type":"string", "description":"기업 주소"},
            },
        },
        "SubsidiaryCompany": {
            "description": "모기업과 종속관계를 가지는 기업",
            "properties": {
                "name": {"type":"string", "description":"종속기업명"},
                "name_norm": {"type":"string", "description":"기업명 매칭을 위한 정규화 이름"},
                "address": {"type":"string", "description":"종속기업 주소"},
                "business_content": {"type":"string", "description":"종속기업의 주요 사업 내용"},
                "domestic": {"type":"string", "description":"국내 또는 해외 구분"},
            },
        },
        "Region": {
            "description": "기업이 위치한 지역",
            "properties": {"name": {"type":"string", "description":"지역명"}},
        },
        "Industry": {
            "description": "기업이 영위하거나 관련된 업종",
            "properties": {"name": {"type":"string", "description":"업종명"}},
        },
    },
    "relationships": {
        "AFFILIATED_WITH": {
            "description": "두 모기업이 같은 기업집단 또는 계열 관계에 있음을 의미",
            "source":"ParentCompany", "target":"ParentCompany",
        },
        "HAS_SUBSIDIARY": {
            "description": "모기업이 해당 종속기업과 종속 관계에 있음을 의미",
            "source":"ParentCompany", "target":"SubsidiaryCompany",
        },
        "LOCATED_IN": {
            "description": "기업이 해당 지역에 위치함을 의미",
            "signatures":[
                {"source":"ParentCompany","target":"Region"},
                {"source":"SubsidiaryCompany","target":"Region"},
            ],
        },
        "IN_INDUSTRY": {
            "description": "기업이 해당 업종을 영위하거나 관련됨을 의미",
            "signatures":[
                {"source":"ParentCompany","target":"Industry"},
                {"source":"SubsidiaryCompany","target":"Industry"},
            ],
        },
    },
}

RELATION_SIGNATURES = {
    "AFFILIATED_WITH":[("ParentCompany","ParentCompany","통합 CSV의 top_crno와 affiliate_crno")],
    "HAS_SUBSIDIARY":[("ParentCompany","SubsidiaryCompany","모기업과 종속기업 매칭")],
    "LOCATED_IN":[
        ("ParentCompany","Region","모기업 또는 계열회사의 지역"),
        ("SubsidiaryCompany","Region","종속기업의 지역"),
    ],
    "IN_INDUSTRY":[
        ("ParentCompany","Industry","모기업 또는 계열회사의 SIC 업종"),
        ("SubsidiaryCompany","Industry","종속기업의 주요 사업 내용"),
    ],
}

def build_ontology_block(signatures):
    lines = ["[허용된 관계 시그니처]"]
    for relation, signature_list in signatures.items():
        for source_type, target_type, criterion in signature_list:
            lines.append(f"- {relation}: ({source_type}) -> ({target_type}) # {criterion}")
    return "\n".join(lines)

print(json.dumps(ONTOLOGY_SCHEMA, ensure_ascii=False, indent=2))
print(build_ontology_block(RELATION_SIGNATURES))


{
  "nodes": {
    "ParentCompany": {
      "description": "계열관계 또는 종속관계의 기준이 되는 법인 기업",
      "properties": {
        "crno": {
          "type": "string",
          "description": "법인등록번호"
        },
        "name": {
          "type": "string",
          "description": "기업명"
        },
        "address": {
          "type": "string",
          "description": "기업 주소"
        }
      }
    },
    "SubsidiaryCompany": {
      "description": "모기업과 종속관계를 가지는 기업",
      "properties": {
        "name": {
          "type": "string",
          "description": "종속기업명"
        },
        "name_norm": {
          "type": "string",
          "description": "기업명 매칭을 위한 정규화 이름"
        },
        "address": {
          "type": "string",
          "description": "종속기업 주소"
        },
        "business_content": {
          "type": "string",
          "description": "종속기업의 주요 사업 내용"
        },
        "domestic": {
          "type": "string",
          "description": "국내 또는 해외 구분"
        }
      }
  

In [2]:
# CSV의 빈 값과 HTML 엔티티를 제거하고 공백을 표준화한다.
def text(value):
    if value is None:
        return ""
    value = str(value)
    if value.lower() in {"nan","none","nat"}:
        return ""
    value = html.unescape(value).replace("&cr;", " ")
    return re.sub(r"\s+", " ", value).strip()

# 법인등록번호에서 숫자만 남기고, 0만 있는 값은 빈 값으로 처리한다.
def clean_crno(value):
    digits = re.sub(r"\D", "", text(value))
    return "" if not digits or set(digits) == {"0"} else digits

# 세미콜론 등으로 묶인 복수 crno를 각각의 ID로 분리한다.
def split_ids(value):
    value = text(value)
    if not value:
        return []
    parts = re.split(r"\s*;\s*|\s*,\s*|\s*\|\s*|\s+", value)
    return list(dict.fromkeys(x for x in (clean_crno(v) for v in parts) if x))

# 지역과 같은 범주형 값을 각각의 값으로 나누는다.
def split_values(value):
    value = text(value)
    if not value:
        return []
    parts = re.split(r"\s*;\s*|\s*\|\s*|\r?\n", value)
    return list(dict.fromkeys(x for x in (text(v) for v in parts) if x))

# 이름과 주소를 비교할 때 공백과 특수문자 차이를 제거한 정규화 키를 만든다.
def normalize_key(value):
    value = text(value).lower()
    return re.sub(r"[^0-9a-z가-힣]", "", value)

# crno는 문자열로 읽어야 앞자리 0이 유지된다. 빈 칸은 모두 빈 문자열로 바꾼다.
corp = pd.read_csv(CORP_PATH, dtype=str, encoding="utf-8-sig").fillna("")
integrated = pd.read_csv(INTEGRATED_PATH, dtype=str, encoding="utf-8-sig").fillna("")
corp["crno"] = corp["crno"].map(clean_crno)
corp = corp[corp["crno"].ne("")].drop_duplicates("crno", keep="last")
CORP_IDS = set(corp["crno"])
CORP_INFO = corp.set_index("crno", drop=False).to_dict("index")

nodes = {}
triples = []

# 반복된 노드는 ID를 기준으로 하나로 합친다. 빈 속성은 다른 행의 값으로 보완한다.
def upsert_node(node_id, node_type, properties):
    record = nodes.setdefault(
        (node_id, node_type),
        {"id":node_id, "type":node_type, "properties":{}},
    )
    for key, value in properties.items():
        value = text(value)
        if value and not record["properties"].get(key):
            record["properties"][key] = value
    return node_id

# 기업개요 CSV에 존재하는 crno만 ParentCompany 노드로 생성한다.
def parent_node(crno, fallback_name="", fallback_address=""):
    if crno not in CORP_IDS:
        return None
    info = CORP_INFO.get(crno, {})
    name = text(info.get("corpNm")) or text(info.get("enpPbanCmpyNm")) or text(fallback_name)
    address = " ".join(
        x for x in [text(info.get("enpBsadr")), text(info.get("enpDtadr"))] if x
    ) or text(fallback_address)
    return upsert_node(
        f"parent_company:{crno}", "ParentCompany",
        {"crno":crno, "name":name, "address":address},
    )

# 트리플과 출처 행 번호를 함께 저장해 추후 검증과 추적이 가능하게 한다.
def emit(subject, subject_type, relation, obj, object_type, row_no, case, evidence=""):
    triples.append({
        "subject":subject, "subject_type":subject_type, "relation":relation,
        "object":obj, "object_type":object_type,
        "source_case":text(case), "source_row":int(row_no),
        "evidence":text(evidence),
    })

# 필터링한 기업개요의 crno 전체를 ParentCompany 노드의 기준으로 사용한다.
# 필터링한 기업개요의 crno 전체를 ParentCompany 노드의 기준으로 사용한다.
for crno in sorted(CORP_IDS):
    parent_node(crno)

# 통합 CSV를 행 단위로 순회하며 계열관계와 종속관계 트리플을 생성한다.
for row_no, row in integrated.iterrows():
    case = row.get("case", "")
    top_ids = split_ids(row.get("top_crno", ""))
    affiliate_id = clean_crno(row.get("affiliate_crno", ""))

    for top_id in top_ids:
        parent_node(top_id, row.get("top_corpNm",""), row.get("top_addr",""))
    parent_node(affiliate_id, row.get("affiliate_corpNm",""), row.get("affiliate_addr",""))

    if affiliate_id in CORP_IDS:
        for top_id in top_ids:
            if top_id in CORP_IDS and top_id != affiliate_id:
                emit(
                    f"parent_company:{top_id}", "ParentCompany", "AFFILIATED_WITH",
                    f"parent_company:{affiliate_id}", "ParentCompany",
                    row_no, case, "top_crno -> affiliate_crno",
                )

    # case3은 affiliate가 종속기업의 직접 모기업이므로 affiliate를 우선한다. 없으면 top_crno를 사용한다.
    subsidiary_name = text(row.get("subsidiary_name",""))
    subsidiary_address = text(row.get("subsidiary_addr",""))
    if subsidiary_name or subsidiary_address:
        # 모기업 crno를 ID에 넣지 않고 종속기업을 전역 하나의 노드로 구분한다.
        name_norm = normalize_key(row.get("name_norm","")) or normalize_key(subsidiary_name)
        address_norm = normalize_key(subsidiary_address)
        domestic_norm = normalize_key(row.get("domestic",""))
        business_norm = normalize_key(row.get("subsidiary_bizCtt",""))
        # 이름과 주소가 모두 있으면 같은 종속기업으로 합치고, 주소가 없으면 보조 키를 추가한다.
        if name_norm and address_norm:
            subsidiary_key = f"name:{name_norm}|address:{address_norm}"
        elif name_norm and (domestic_norm or business_norm):
            subsidiary_key = f"name:{name_norm}|fallback:{domestic_norm}|{business_norm}"
        elif name_norm:
            subsidiary_key = f"name:{name_norm}|row:{row_no}"
        else:
            subsidiary_key = f"address:{address_norm}|row:{row_no}" if address_norm else ""
        immediate_parents = [affiliate_id] if affiliate_id else top_ids
        if subsidiary_key:
            subsidiary_id = f"subsidiary:{subsidiary_key}"
            upsert_node(
                subsidiary_id, "SubsidiaryCompany",
                {
                    "name":subsidiary_name,
                    "name_norm":name_norm,
                    "address":subsidiary_address,
                    "business_content":row.get("subsidiary_bizCtt",""),
                    "domestic":row.get("domestic",""),
                },
            )
            for parent_id in dict.fromkeys(immediate_parents):
                if parent_id in CORP_IDS:
                    emit(
                        f"parent_company:{parent_id}", "ParentCompany", "HAS_SUBSIDIARY",
                        subsidiary_id, "SubsidiaryCompany",
                        row_no, case, subsidiary_name,
                    )

            for region in split_values(row.get("subsidiary_region","")):
                region_key = normalize_key(region)
                if region_key:
                    region_id = f"region:{region_key}"
                    upsert_node(region_id, "Region", {"name":region})
                    emit(
                        subsidiary_id, "SubsidiaryCompany", "LOCATED_IN",
                        region_id, "Region", row_no, case, region,
                    )

            business = text(row.get("subsidiary_bizCtt",""))
            business_key = normalize_key(business)
            if business_key:
                industry_id = f"industry:{business_key}"
                upsert_node(industry_id, "Industry", {"name":business})
                emit(
                    subsidiary_id, "SubsidiaryCompany", "IN_INDUSTRY",
                    industry_id, "Industry", row_no, case, business,
                )

    # top_crno가 하나인 행에서만 top_region을 해당 기업에 연결한다. 여러 crno가 함께 있는 행은 제외한다.
    if len(top_ids) == 1 and top_ids[0] in CORP_IDS:
        for region in split_values(row.get("top_region","")):
            region_key = normalize_key(region)
            if region_key:
                region_id = f"region:{region_key}"
                upsert_node(region_id, "Region", {"name":region})
                emit(
                    f"parent_company:{top_ids[0]}", "ParentCompany", "LOCATED_IN",
                    region_id, "Region", row_no, case, region,
                )

    if affiliate_id in CORP_IDS:
        for region in split_values(row.get("affiliate_region","")):
            region_key = normalize_key(region)
            if region_key:
                region_id = f"region:{region_key}"
                upsert_node(region_id, "Region", {"name":region})
                emit(
                    f"parent_company:{affiliate_id}", "ParentCompany", "LOCATED_IN",
                    region_id, "Region", row_no, case, region,
                )

    # 모기업과 계열회사의 SIC 업종을 Industry 노드와 연결한다.
    for parent_id, value in (
        [(x, row.get("top_sicNm","")) for x in top_ids]
        + [(affiliate_id, row.get("affiliate_sicNm",""))]
    ):
        value = text(value)
        value_key = normalize_key(value)
        if parent_id in CORP_IDS and value_key:
            industry_id = f"industry:{value_key}"
            upsert_node(industry_id, "Industry", {"name":value})
            emit(
                f"parent_company:{parent_id}", "ParentCompany", "IN_INDUSTRY",
                industry_id, "Industry", row_no, case, value,
            )

print(f"읽은 기업개요 행 수={len(corp):,}, 통합 데이터 행 수={len(integrated):,}")
print(f"중복 제거 전 노드 수={len(nodes):,}, 중복 제거 전 트리플 수={len(triples):,}")


읽은 기업개요 행 수=562, 통합 데이터 행 수=17,398
중복 제거 전 노드 수=10,350, 중복 제거 전 트리플 수=49,689


In [7]:
# 관계의 주체 및 객체 종류가 온톨로지 시그니처와 일치하는지 확인한다.
def check_signature(triple):
    allowed = RELATION_SIGNATURES.get(triple["relation"], [])
    actual = (triple["subject_type"], triple["object_type"])
    valid_pairs = {(source, target) for source, target, _ in allowed}
    if actual not in valid_pairs:
        return False, f"{triple['relation']}: {actual} is not allowed"
    return True, ""

# 생성한 트리플을 표 형식으로 변환한다.
triples_df = pd.DataFrame(triples)
if triples_df.empty:
    triples_df = pd.DataFrame(columns=[
        "subject","subject_type","relation","object","object_type",
        "source_case","source_row","evidence",
    ])
else:
    triples_df = triples_df.drop_duplicates(
        subset=["subject","relation","object"], keep="first"
    ).reset_index(drop=True)

# 허용되지 않은 관계 시그니처가 발견되면 출력 전에 즉시 확인한다.
errors = []
for record in triples_df.to_dict("records"):
    valid, reason = check_signature(record)
    if not valid:
        errors.append(reason)
assert not errors, errors[:10]

# 노드 ID, 노드 종류, 속성을 후속 적재에 쓰기 쉽도록 표 형식으로 변환한다.
nodes_df = pd.DataFrame([
    {"id":value["id"], "type":value["type"], **value["properties"]}
    for value in nodes.values()
])
if not nodes_df.empty:
    nodes_df = nodes_df.sort_values(["type","id"]).reset_index(drop=True)

# 관계 유형별 최종 트리플 개수를 요약해 검수 결과를 확인한다.
summary = (
    triples_df.groupby(["relation","subject_type","object_type"], dropna=False)
    .size().reset_index(name="count")
    if not triples_df.empty
    else pd.DataFrame(columns=["relation","subject_type","object_type","count"])
)

# 교안의 JSONL 파일처럼 한 줄에 노드 혹은 트리플 하나만 저장한다.
# 노드 JSONL은 properties 안에 속성을 넣고, 트리플 JSONL은 subject/object와 노드 종류를 함께 저장한다.
NODE_JSONL_OUTPUT = CLEAN_DIR / "기업관계_노드.jsonl"
TRIPLE_JSONL_OUTPUT = CLEAN_DIR / "기업관계_트리플.jsonl"

with NODE_JSONL_OUTPUT.open("w", encoding="utf-8", newline="\n") as file:
    for node in nodes.values():
        file.write(json.dumps(node, ensure_ascii=False) + "\n")

with TRIPLE_JSONL_OUTPUT.open("w", encoding="utf-8", newline="\n") as file:
    for triple in triples_df.to_dict("records"):
        file.write(json.dumps(triple, ensure_ascii=False) + "\n")

print(f"노드 JSONL 저장 완료: {NODE_JSONL_OUTPUT}")
print(f"트리플 JSONL 저장 완료: {TRIPLE_JSONL_OUTPUT}")


검증 완료 노드 수=10,350, 중복 제거 트리플 수=19,329
       relation      subject_type       object_type  count
AFFILIATED_WITH     ParentCompany     ParentCompany    478
 HAS_SUBSIDIARY     ParentCompany SubsidiaryCompany   4637
    IN_INDUSTRY     ParentCompany          Industry    739
    IN_INDUSTRY SubsidiaryCompany          Industry   7183
     LOCATED_IN     ParentCompany            Region    562
     LOCATED_IN SubsidiaryCompany            Region   5730
노드 CSV 저장 완료: c:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\data\clean\기업관계_노드.csv
트리플 CSV 저장 완료: c:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\data\clean\기업관계_트리플.csv
노드 JSONL 저장 완료: c:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\data\clean\기업관계_노드.jsonl
트리플 JSONL 저장 완료: c:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\data\clean\기업관계_트리플.jsonl


In [6]:
triples

[{'subject': 'parent_company:1101110000086',
  'subject_type': 'ParentCompany',
  'relation': 'AFFILIATED_WITH',
  'object': 'parent_company:1101110014764',
  'object_type': 'ParentCompany',
  'source_case': '1',
  'source_row': 0,
  'evidence': 'top_crno -> affiliate_crno'},
 {'subject': 'parent_company:1101110000086',
  'subject_type': 'ParentCompany',
  'relation': 'LOCATED_IN',
  'object': 'region:서울',
  'object_type': 'Region',
  'source_case': '1',
  'source_row': 0,
  'evidence': '서울'},
 {'subject': 'parent_company:1101110014764',
  'subject_type': 'ParentCompany',
  'relation': 'LOCATED_IN',
  'object': 'region:서울',
  'object_type': 'Region',
  'source_case': '1',
  'source_row': 0,
  'evidence': '서울'},
 {'subject': 'parent_company:1101110000086',
  'subject_type': 'ParentCompany',
  'relation': 'IN_INDUSTRY',
  'object': 'industry:백화점',
  'object_type': 'Industry',
  'source_case': '1',
  'source_row': 0,
  'evidence': '백화점'},
 {'subject': 'parent_company:1101110014764',
  'su

In [8]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

Neo4j 연결: bolt://localhost:7687


In [10]:
# [제공 코드] 실습 전용 DB 초기화: 이 데이터베이스를 통째로 비웁니다.
# ⚠️ 가리는 것 없이 **노드·관계·제약을 전부** 지웁니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH 는 노드에 붙은 관계까지 함께 지웁니다
# 제약조건은 노드를 지워도 남습니다. 이름을 조회해 하나씩 DROP 합니다
for _c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _c["name"] + " IF EXISTS")
print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

초기화 완료: bolt://localhost:7687 · 남은 노드: 0


In [11]:
import json
from collections import defaultdict
from pathlib import Path

# JSONL 파일 위치
CLEAN_DIR = Path("data/clean")
NODE_JSONL_PATH = CLEAN_DIR / "기업관계_노드.jsonl"
TRIPLE_JSONL_PATH = CLEAN_DIR / "기업관계_트리플.jsonl"

BATCH_SIZE = 1000

NODE_TYPES = {
    "ParentCompany",
    "SubsidiaryCompany",
    "Region",
    "Industry",
}

ALLOWED_SIGNATURES = {
    ("AFFILIATED_WITH", "ParentCompany", "ParentCompany"),
    ("HAS_SUBSIDIARY", "ParentCompany", "SubsidiaryCompany"),
    ("LOCATED_IN", "ParentCompany", "Region"),
    ("LOCATED_IN", "SubsidiaryCompany", "Region"),
    ("IN_INDUSTRY", "ParentCompany", "Industry"),
    ("IN_INDUSTRY", "SubsidiaryCompany", "Industry"),
}


def read_jsonl(path):
    """JSONL 파일을 한 줄씩 읽어 dict 리스트로 반환한다."""
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def run_in_batches(query, rows, batch_size=BATCH_SIZE):
    """많은 데이터를 한 번에 보내지 않고 일정 개수씩 나누어 적재한다."""
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        run_cypher(query, rows=batch)


# 1. 노드 JSONL 읽기
node_rows = read_jsonl(NODE_JSONL_PATH)

nodes_by_type = defaultdict(list)

for row in node_rows:
    node_type = row["type"]

    if node_type not in NODE_TYPES:
        raise ValueError(f"허용되지 않은 노드 타입: {node_type}")

    nodes_by_type[node_type].append(row)


# 2. 노드 ID 유일성 제약조건 생성
# 노드의 id는 기업의 crno 또는 종속기업의 합성 ID이다.
for node_type in NODE_TYPES:
    constraint_name = f"{node_type.lower()}_id_unique"

    query = f"""
    CREATE CONSTRAINT {constraint_name} IF NOT EXISTS
    FOR (n:{node_type})
    REQUIRE n.id IS UNIQUE
    """

    run_cypher(query)


# 3. 노드 적재
# label은 파라미터로 전달할 수 없으므로 노드 타입별로 쿼리를 생성한다.
for node_type, rows in nodes_by_type.items():
    query = f"""
    UNWIND $rows AS row
    MERGE (n:{node_type} {{id: row.id}})
    SET n += row.properties
    SET n.id = row.id
    """

    run_in_batches(query, rows)


# 4. 트리플 JSONL 읽기
triple_rows = read_jsonl(TRIPLE_JSONL_PATH)

triples_by_signature = defaultdict(list)

for row in triple_rows:
    signature = (
        row["relation"],
        row["subject_type"],
        row["object_type"],
    )

    if signature not in ALLOWED_SIGNATURES:
        raise ValueError(f"허용되지 않은 관계 시그니처: {signature}")

    triples_by_signature[signature].append(row)


# 5. 관계 적재
# 관계 타입과 노드 label은 허용된 시그니처에서 검증된 값만 쿼리에 삽입한다.
for (relation, subject_type, object_type), rows in triples_by_signature.items():
    query = f"""
    UNWIND $rows AS row

    MATCH (subject:{subject_type} {{id: row.subject}})
    MATCH (object:{object_type} {{id: row.object}})

    MERGE (subject)-[r:{relation}]->(object)

    SET r.source_case = row.source_case,
        r.source_row = row.source_row,
        r.evidence = row.evidence
    """

    run_in_batches(query, rows)


print(f"노드 적재 대상: {len(node_rows):,}개")
print(f"트리플 적재 대상: {len(triple_rows):,}개")

노드 적재 대상: 10,350개
트리플 적재 대상: 19,329개


## 생성되는 관계

(ParentCompany, AFFILIATED_WITH, ParentCompany)
(ParentCompany, HAS_SUBSIDIARY, SubsidiaryCompany)
(ParentCompany, LOCATED_IN, Region)
(SubsidiaryCompany, LOCATED_IN, Region)
(ParentCompany, IN_INDUSTRY, Industry)
(SubsidiaryCompany, IN_INDUSTRY, Industry)

종속기업에 실제 crno가 제공되면 합성 ID를 해당 crno 기반 ID로 교체할 수 있다. 이름과 주소가 모두 비어 있는 종속기업은 노드로 만들지 않는다.
